# DWBI Week 3 — Fact & Dimension Tables
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

PakFreight's flat `Shipments_Unified` (34 rows, 578,650 kg) becomes **one fact table + five dimensions** — and the totals do not move by a single kilogram.

## T7 — Reconcile: 13 / 13

| # | Check | Result |
|---|---|---|
| 1 | FactShipment rows | 34 ✔ |
| 2 | Total WeightKG | 578,650 ✔ |
| 3 | Total FreightCostPKR | 4,808,000 ✔ |
| 4 | Shipment legs | 34 ✔ |
| 5 | Distinct ShipmentID | 34 ✔ |
| 6 | DimCity rows | 14 ✔ |
| 7 | DimRoute rows | 20 ✔ |
| 8 | DimDate rows | 28 ✔ |
| 9 | Sindh / Marine | 169,000 ✔ (206,500 would mean Gwadar→Karachi) |
| 10 | Punjab / Road | 44,100 ✔ (Islamabad is ICT, not Punjab) |
| 11 | Sindh, all modes | 484,350 ✔ |
| 12 | Unknown-key refs | 0 ✔ |
| 13 | Cost per KG | 8.31 ✔ (8.06 = averaged ratio) |

**The last question** — one thing no check would ever catch: the checks prove *internal arithmetic*,
not external truth. If every weight in the source files was understated 10% at the source, all thirteen
still pass and the model is still wrong. Reconciliation catches reshaping errors; only the business can
catch a lie told consistently.

In [1]:
import pandas as pd
f = pd.read_csv("data/FactShipment.csv"); c = pd.read_csv("data/DimCity.csv")
k2p = dict(zip(c.CityKey, c.Province))
prov = f.OriginCityKey.map(k2p); mode = f.ModeKey.map({1:"Marine",2:"Road",3:"Road",2:"Rail"})
mode = f.ModeKey.map({1:"Marine",2:"Rail",3:"Road"})
mx = pd.crosstab(prov, mode, values=f.WeightKG, aggfunc="sum").fillna(0).astype(int)
print(mx.to_string())
print("\nGrand total:", f"{mx.values.sum():,} kg  -> 13/13")

ModeKey        Marine    Rail   Road
OriginCityKey                       
Balochistan     37500       0   6600
Islamabad ICT       0       0   6100
Punjab              0       0  44100
Sindh          169000  295250  20100

Grand total: 578,650 kg  -> 13/13
